# aether — voice model and limited adaptation
Full manual scenario: Git → environment → data → audio and text → baseline loss →
5-step pilot → resume to 20 steps → comparison → saved adapter.

The full scenario requires a **remote managed paid Colab runtime with an A100, 40GB
VRAM or larger**. An L4 with 24GB is intended for inference only. A T4 with 15GB,
per a previous report, is not supported by this BF16 profile: it fails before the
weights are loaded. These are admission thresholds, not a guarantee against OOM. On
OOM, the frame count should be reduced to 32, or more memory should be used; the
checkpoint from the previous step is preserved.

The stage budget is 500 units. The notebook cannot debit or check this budget via an
API: consumption must be monitored in the interface. Training is enabled explicitly
via RUN_TRAINING=True. The data consists of short English reading recordings: this
adapts audio generation, not substantive dialogue capability. Listening to the
before/after audio is required to assess the result.

In [ ]:
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

CONFIRM_REMOTE_PAID_COLAB = False
RUN_TRAINING = False
RUN_ID = "english-demo-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
DRIVE_ROOT = "/content/drive/MyDrive/aether"
REPO_URL = "https://github.com/karl4th/aether-v2.git"
GIT_REF = "main"  # For resume use source_revision from the saved run.json.
UV_VERSION = "0.12.13"
BUDGET_UNITS = 500
RESUME_CHECKPOINT = ""  # Optional COMPLETE checkpoint directory on Drive.
UPLOAD_QUESTION = False  # True: upload your English question as WAV/FLAC.

## Environment and exact checkout
Confirmation of the remote paid environment selection is required. Availability of
the library in the isolated uv environment is not required. For a private GitHub
repository, a read-only GITHUB_TOKEN Secret is used. The token is not stored in the
URL or in the Git configuration.

In [ ]:
if not CONFIRM_REMOTE_PAID_COLAB:
    raise RuntimeError("Confirm managed remote paid Colab; local runtime is forbidden")
__import__("google.colab")
if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Select a remote Colab Linux GPU runtime")

In [ ]:
import base64
import os
import tempfile

from google.colab import userdata


def checkout_source(repo_url, git_ref, workspace, git_env=None):
    if not git_ref or git_ref.startswith("-"):
        raise ValueError("Expected a Git branch, tag or commit SHA")
    project = Path(tempfile.mkdtemp(prefix="aether-src-", dir=workspace))
    env = dict(os.environ if git_env is None else git_env)
    env["GIT_TERMINAL_PROMPT"] = "0"

    def git(*args):
        result = subprocess.run(
            ["git", *args],
            cwd=project,
            env=env,
            check=True,
            capture_output=True,
            text=True,
            timeout=180,
        )
        return result.stdout.strip()

    git("init", "--quiet")
    git("remote", "add", "origin", repo_url)
    git("fetch", "--depth=1", "origin", git_ref)
    revision = git("rev-parse", "FETCH_HEAD^{commit}")
    git("checkout", "--detach", revision)
    for name in ("pyproject.toml", "uv.lock", ".python-version"):
        if not (project / name).is_file():
            raise ValueError("Missing required project file: " + name)
    return project, revision


# Optional read-only GitHub credential; never printed or stored in Git config.
git_env = os.environ.copy()
try:
    token = userdata.get("GITHUB_TOKEN")
except userdata.SecretNotFoundError:
    token = None
if token:
    if REPO_URL != "https://github.com/karl4th/aether-v2.git":
        raise ValueError("Credential use is restricted to the aether repository")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": "Authorization: Basic " + auth,
        }
    )
    del auth
try:
    PROJECT, SOURCE_REVISION = checkout_source(REPO_URL, GIT_REF, "/content", git_env)
finally:
    git_env.clear()
    del token
print("Source commit:", SOURCE_REVISION)
print("Project:", PROJECT)

## Installation only in the remote environment
uv uses the lock file, Python 3.12.14, and a separate .venv. The weights are loaded
later, after the resource check. Each model invocation runs as a separate process:
GPU memory is freed between baseline, training, and evaluation.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==" + UV_VERSION], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(
    UV + ["sync", "--locked", "--no-dev", "--group", "model", "--python", "3.12.14"],
    cwd=PROJECT,
    check=True,
)


def package_run(*arguments):
    result = subprocess.run(
        UV + ["run", "--locked", "--no-dev", "--group", "model", *map(str, arguments)],
        cwd=PROJECT,
        text=True,
        capture_output=True,
    )
    if result.returncode:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("aether operation failed; inspect the diagnostic above")
    return result


def command_json(*arguments):
    result = package_run("aether", *arguments)
    if result.stderr:
        print(result.stderr[-4000:])
    return json.loads(result.stdout.splitlines()[-1])


print(
    package_run(
        "aether", "train", "--config", "configs/training/colab_lora.json", "--validate-only"
    ).stdout
)

## Runtime permit and storage
The permit is bound to the boot ID and the live notebook process, and is valid for
up to 12 hours. It guards against accidental local execution; it is not
cryptographic proof of the billing tier. The explicit confirmation of the paid
runtime is supplemented by a check of several environment signals. Results and
checkpoints are written to a new run on Google Drive.

In [ ]:
import hashlib
import os
import time

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted")
PERMIT = PROJECT / "runtime-permit.json"
now = time.time()
PERMIT.write_text(
    json.dumps(
        {
            "schema_version": 1,
            "boot_id": Path("/proc/sys/kernel/random/boot_id").read_text().strip(),
            "kernel_pid": os.getpid(),
            "created_at": now,
            "expires_at": now + 12 * 3600,
            "user_confirmed_remote_paid": CONFIRM_REMOTE_PAID_COLAB,
            "allow_training": RUN_TRAINING,
            "budget_units": BUDGET_UNITS,
            "source_revision": SOURCE_REVISION,
        }
    )
)
result = package_run(
    "python",
    "-c",
    "import sys; from pathlib import Path; from aether.storage import create_run; "
    "print(create_run(Path(sys.argv[1]),sys.argv[2]))",
    DRIVE_ROOT,
    RUN_ID,
)
RUN_DIR = Path(result.stdout.strip())
report = json.loads(
    package_run(
        "python",
        "-c",
        "import json; from aether.preflight import collect_preflight; "
        "print(json.dumps(collect_preflight()))",
    ).stdout
)
report.update(
    source_revision=SOURCE_REVISION,
    source_repository=REPO_URL,
    uv_lock_sha256=hashlib.sha256((PROJECT / "uv.lock").read_bytes()).hexdigest(),
    budget_units=BUDGET_UNITS,
    training_requested=RUN_TRAINING,
)
(RUN_DIR / "run.json").write_text(json.dumps(report, indent=2))
print(report["gpu"])
package_run(
    "python",
    "-c",
    "import sys; from aether.remote import require_remote_runtime; "
    "require_remote_runtime(permit_path=sys.argv[1]); import torch; "
    "from aether.backend import check_resources; "
    "check_resources(torch, training=sys.argv[2]=='True')",
    PERMIT,
    str(RUN_TRAINING),
)
print("Resource admission passed; actual peak memory is checked by real operations below.")
backend_check = (
    "from aether.backend_contract import validate_backend_api; print(validate_backend_api())"
)
print(package_run("python", "-c", backend_check).stdout)

## English data
The official archives of a small corpus are downloaded, with verification against
published checksums. 16 training and 4 evaluation recordings of 2-5 seconds are
selected; speakers do not overlap between the two sets. The original audio and the
CC BY 4.0 terms are preserved. The manifest and attribution are copied to Drive; the
working audio remains on the VM disk and is reproducibly re-downloaded for resume.
The SHA256 hashes of the selected audio files are part of the dataset identity.

In [ ]:
import shutil

DATA_ROOT = Path("/content/aether-data")
data_result = command_json(
    "prepare-data",
    "--output",
    DATA_ROOT,
    "--permit",
    PERMIT,
    "--max-train",
    "16",
    "--max-eval",
    "4",
    "--max-seconds",
    "5",
)
DATASET = Path(data_result["manifest"])
shutil.copy2(DATASET, RUN_DIR / "dataset.json")
shutil.copy2(DATA_ROOT / "ATTRIBUTION.txt", RUN_DIR / "DATA_ATTRIBUTION.txt")
raw_dataset = json.loads(DATASET.read_text())
INPUT_AUDIO = DATA_ROOT / raw_dataset["evaluation"][0]["audio_path"]
if UPLOAD_QUESTION:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one short English question as WAV or FLAC")
    INPUT_AUDIO = PROJECT / "question.wav"
    INPUT_AUDIO.write_bytes(next(iter(uploaded.values())))
print(data_result)

## Inference with listening
A fixed revision of the matched component set is used: the voice model, the base
codec, and the tokenizer. About 16GB is downloaded on the first operation;
subsequent runs use the local cache. Weight hashes, the seed, revisions, and
parameters are recorded. Without an upload, the input is a held-out reading
recording; for a question/answer demo, UPLOAD_QUESTION above should be enabled.

In [ ]:
from IPython.display import Audio, display

baseline = command_json(
    "infer",
    "--config",
    "configs/model/remote.json",
    "--permit",
    PERMIT,
    "--input",
    INPUT_AUDIO,
    "--output",
    RUN_DIR / "baseline",
)
print("Input:")
display(Audio(filename=str(INPUT_AUDIO)))
print(baseline["text"])
display(Audio(filename=baseline["audio"]))

## Evaluation before adaptation
Teacher-forced audio cross-entropy/perplexity is computed over held-out speakers.
Text without word timestamps is excluded from the loss: it does not receive a
fabricated alignment. This metric does not substitute for an assessment of dialogue
quality or intelligibility.

In [ ]:
before = command_json(
    "evaluate",
    "--config",
    "configs/training/colab_lora.json",
    "--permit",
    PERMIT,
    "--dataset",
    DATASET,
    "--output",
    RUN_DIR / "before.json",
)
print(before)

## Pilot → save → new process → resume
This step runs only when RUN_TRAINING=True. The base model and the base codec are
frozen; a rank-8 adapter is applied to the last temporal FFN, with 64 frames,
batch=1, 20 steps, and a learning rate of 1e-4. The pilot executes 5 steps of the
overall plan and saves the optimizer/scheduler/RNG state; a second process then
restores the checkpoint and continues to step 20.
The training loop is capped at 30 minutes (excluding loading time); if the limit is
reached, an incomplete result is saved.
For a new session, RESUME_CHECKPOINT and the original GIT_REF should be set; the
parameters should not be changed.
Completing 20 steps within the given time cannot be guaranteed before measurement on
the dedicated GPU.

In [ ]:
FINAL_CHECKPOINT = None
if RUN_TRAINING:
    if RESUME_CHECKPOINT:
        checkpoint = RESUME_CHECKPOINT
    else:
        pilot = command_json(
            "train",
            "--config",
            "configs/training/colab_lora.json",
            "--permit",
            PERMIT,
            "--dataset",
            DATASET,
            "--output",
            RUN_DIR,
            "--stop-after-steps",
            "5",
        )
        checkpoint = pilot["checkpoint"]
        (RUN_DIR / "pilot.json").write_text(json.dumps(pilot, indent=2))
    trained = command_json(
        "train",
        "--config",
        "configs/training/colab_lora.json",
        "--permit",
        PERMIT,
        "--dataset",
        DATASET,
        "--output",
        RUN_DIR,
        "--resume",
        checkpoint,
    )
    FINAL_CHECKPOINT = trained["checkpoint"]
    (RUN_DIR / "training.json").write_text(json.dumps(trained, indent=2))
    print(
        {"completed": trained["completed"], "step": trained["step"], "checkpoint": FINAL_CHECKPOINT}
    )
else:
    print("Baseline completed. Training was not requested; no optimizer was constructed.")

## After adaptation: the same evaluation and the same audio
The checkpoint is applied to the same revision of the base model; it is not merged
with weights from another source. The loss values are compared and the resulting
audio is reviewed by listening. A decrease in loss does not prove an improvement in
conversational quality; the limited pilot serves as a check of the pipeline.

In [ ]:
if FINAL_CHECKPOINT:
    after = command_json(
        "evaluate",
        "--config",
        "configs/training/colab_lora.json",
        "--permit",
        PERMIT,
        "--dataset",
        DATASET,
        "--checkpoint",
        FINAL_CHECKPOINT,
        "--output",
        RUN_DIR / "after.json",
    )
    candidate = command_json(
        "infer",
        "--config",
        "configs/model/remote.json",
        "--permit",
        PERMIT,
        "--input",
        INPUT_AUDIO,
        "--checkpoint",
        FINAL_CHECKPOINT,
        "--output",
        RUN_DIR / "adapted",
    )
    print({"before_audio_ce": before["audio_ce"], "after_audio_ce": after["audio_ce"]})
    print(candidate["text"])
    display(Audio(filename=candidate["audio"]))

## Export and completion
The export contains the adapter, the optimizer/RNG state for resuming, the
manifest, hashes, and attribution. The base weights are not copied again. The same
code revision should be used for resuming; if the identity does not match, loading
is rejected.

In [ ]:
if FINAL_CHECKPOINT:
    export_dir = Path(DRIVE_ROOT) / "exports" / RUN_ID
    package_run(
        "python",
        "-c",
        "import sys; from pathlib import Path; from aether.storage import verify_checkpoint; "
        "verify_checkpoint(Path(sys.argv[1]))",
        FINAL_CHECKPOINT,
    )
    shutil.copytree(FINAL_CHECKPOINT, export_dir)
    package_run(
        "python",
        "-c",
        "import sys; from pathlib import Path; from aether.storage import verify_checkpoint; "
        "verify_checkpoint(Path(sys.argv[1]))",
        export_dir,
    )
    shutil.copy2(PROJECT / "THIRD_PARTY_NOTICES.md", RUN_DIR / "THIRD_PARTY_NOTICES.md")
    print("Export verified:", export_dir)
print("Results:", RUN_DIR)
print("Disconnect the GPU runtime when finished to stop idle resource consumption.")